# An Explainable Machine Learning System for Estimating Help-Seeking Intentions from Dating Violence Experiences Among Sri Lankan University Students
Predictors: D01–D20 (DX responses). Target: professional help-seeking intention H3.
Classes: 0 = low (1–3), 1 = neutral (4), 2 = high (5–7).
Run in Colab in order. Keep the main-study CSV in `data/`; pilot responses are excluded.
Selection: highest development cross-validation macro F1, then ROC AUC. Final testing does not change selection.
If dependencies change, setup restarts the runtime once. After reconnection, run all cells again.

Dating-violence response reference period: past six months.


In [ ]:
import importlib.metadata as metadata
import subprocess, sys, os, signal
versions = {'numpy':'2.2.6','pandas':'2.2.3','scipy':'1.15.3','scikit-learn':'1.7.2','xgboost':'3.0.5','shap':'0.48.0','matplotlib':'3.10.3','joblib':'1.5.1'}
changed = False
for package, version in versions.items():
    try:
        changed |= metadata.version(package) != version
    except metadata.PackageNotFoundError:
        changed = True
if changed:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*[f'{p}=={v}' for p,v in versions.items()]])
    print('Environment installed. Runtime is restarting; run all cells again after reconnection.', flush=True)
    os.kill(os.getpid(), signal.SIGKILL)
else:
    print('Pinned environment ready.')


In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Final Research - C D Jayasinghe')
for folder in ['data', 'models', 'outputs', 'report_figures', 'deployment', 'notebooks']:
    (ROOT / folder).mkdir(parents=True, exist_ok=True)

## 1. Validate and prepare data

In [ ]:
import hashlib, json, platform, re
import importlib.metadata as metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.model_selection import StratifiedGroupKFold, RandomizedSearchCV, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, ConfusionMatrixDisplay, RocCurveDisplay)
from xgboost import XGBClassifier

SEED = 42
FEATURES = [f'D{i:02d}' for i in range(1, 21)]
LABELS = ['Low intention', 'Neutral intention', 'High intention']
files = sorted(p for p in (ROOT / 'data').rglob('*.csv')
               if 'pilot' not in str(p).lower() and p.name != 'prepared.csv')
if len(files) != 1:
    raise ValueError(f'Expected one main-study CSV in data/, found {len(files)}. Keep evaluation files in outputs/.')
source = files[0]
raw = pd.read_csv(source, dtype='string').replace(r'^\s*$', pd.NA, regex=True)
raw = raw.replace('Prefer not to answer', pd.NA)
raw.columns = raw.columns.str.strip()
if not raw.columns.is_unique:
    raise ValueError('Duplicate CSV headers.')

def find_column(pattern):
    found = [c for c in raw.columns if re.match(pattern, c, flags=re.I)]
    if len(found) != 1:
        raise ValueError(f'Ambiguous or missing column: {pattern}')
    return found[0]

columns = {f: find_column(rf'^{f}\s*[.:-]') for f in FEATURES}
h3_column = find_column(r'^H3\s*[.:-]')
consent = find_column(r'^Consent:')
eligibility = find_column(r'^Do you meet all four eligibility criteria')
eligible = raw[consent].eq('Yes, I agree').fillna(False) & raw[eligibility].eq('Yes, all four apply').fillna(False)
study = raw.loc[eligible].copy()
duplicates = int(study.duplicated().sum())
study = study.drop_duplicates()

def parse_scale(series, low, high):
    # Accept an integer alone or an integer followed by the survey's labelled dash.
    text = series.str.strip()
    numeric = pd.to_numeric(text.str.extract(r'^(\d+)(?:\s*[—–-]\s*.+)?$', expand=False), errors='coerce')
    invalid = series.notna() & (numeric.isna() | ~numeric.between(low, high))
    if invalid.any():
        raise ValueError(f'Invalid response in {series.name}: {int(invalid.sum())} rows')
    return numeric.astype(float)

X = pd.DataFrame({f: parse_scale(study[c], 0, 4) for f, c in columns.items()})
h3 = parse_scale(study[h3_column], 1, 7)
keep = h3.notna() & X.notna().any(axis=1)
X, h3 = X.loc[keep].reset_index(drop=True), h3.loc[keep].reset_index(drop=True)
y = pd.Series(np.select([h3.le(3), h3.eq(4)], [0, 1], default=2), name='H3_class')
if 'H3_Group' in study.columns:
    existing = study.loc[keep, 'H3_Group'].str.strip().str.lower().reset_index(drop=True)
    expected = y.map({0: 'low', 1: 'neutral', 2: 'high'})
    if (existing.notna() & existing.ne(expected)).any():
        raise ValueError('Existing H3_Group disagrees with the H3 scoring rule.')
if set(y.unique()) != {0, 1, 2}:
    raise ValueError('All three H3 classes are required.')

# Identical DX patterns stay together, including conflicting H3 labels.
groups = pd.util.hash_pandas_object(X.fillna(-1), index=False).astype(str)
if min(groups.groupby(y).nunique()) < 10:
    raise ValueError('Too few independent patterns in at least one class for the planned validation.')
audit = {'raw_n': len(raw), 'ineligible_n': int((~eligible).sum()),
         'exact_duplicates_removed': duplicates, 'unusable_n': int((~keep).sum()),
         'analysis_n': len(X), 'missing_dx_cells': int(X.isna().sum().sum()),
         'repeated_dx_patterns_n': int(groups.duplicated().sum()),
         'class_counts': {LABELS[k]: int(v) for k, v in y.value_counts().sort_index().items()},
         'source_sha256': hashlib.sha256(source.read_bytes()).hexdigest(),
         'pilot_included': False, 'predictors': FEATURES,
         'target_rule': 'H3 1-3:0; 4:1; 5-7:2'}
(ROOT / 'outputs/data_audit.json').write_text(json.dumps(audit, indent=2))
display(pd.Series(audit).to_frame('Value'))
ax = y.map(dict(enumerate(LABELS))).value_counts().reindex(LABELS).plot.bar(rot=0, color=['#455a64','#b38b31','#287c74'])
ax.set(ylabel='Responses', title='Professional help-seeking intention')
plt.tight_layout(); plt.savefig(ROOT / 'report_figures/class_distribution.png', dpi=200); plt.show()

## 2. Freeze independent test data and cross-validation folds

In [ ]:
outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
train_idx, test_idx = next(outer.split(X, y, groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
train_groups = groups.iloc[train_idx]
assert set(train_groups).isdisjoint(groups.iloc[test_idx])
assert set(y_train) == set(y_test) == {0, 1, 2}
if X_train.notna().sum().eq(0).any():
    raise ValueError('A DX feature is entirely missing in development data.')
cv = list(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED + 1)
          .split(X_train, y_train, train_groups))
for a, b in cv:
    assert set(y_train.iloc[a]) == set(y_train.iloc[b]) == {0, 1, 2}
    assert set(train_groups.iloc[a]).isdisjoint(train_groups.iloc[b])
split = pd.DataFrame({'row': np.arange(len(X)), 'partition': 'development'})
split.loc[test_idx, 'partition'] = 'test'
split.to_csv(ROOT / 'outputs/split_manifest.csv', index=False)
split_summary = pd.crosstab(split['partition'], y.map(dict(enumerate(LABELS))))
split_summary.to_csv(ROOT / 'outputs/split_summary.csv')
display(split_summary)
SCORING = {'accuracy': 'accuracy', 'precision_macro': 'precision_macro',
           'recall_macro': 'recall_macro', 'f1_macro': 'f1_macro',
           'roc_auc_ovr_macro': 'roc_auc_ovr'}

## 3. Train and tune the candidate models

In [ ]:
def pipeline(model, scale=False):
    steps = [('imputer', SimpleImputer(strategy='median', keep_empty_features=True))]
    if scale:
        steps.append(('scale', StandardScaler()))
    return Pipeline(steps + [('model', model)])

candidates = {
    'Logistic Regression': (
        pipeline(LogisticRegression(max_iter=4000, solver='lbfgs', random_state=SEED), True),
        {'model__C': np.logspace(-3, 2, 8), 'model__class_weight': [None, 'balanced']}),
    'Random Forest': (
        pipeline(RandomForestClassifier(random_state=SEED, n_jobs=1)),
        {'model__n_estimators': [200, 400], 'model__max_depth': [3, 6, 10, None],
         'model__min_samples_leaf': [2, 5, 10], 'model__max_features': ['sqrt', 0.7],
         'model__class_weight': [None, 'balanced']}),
    'XGBoost': (
        pipeline(XGBClassifier(objective='multi:softprob', num_class=3, eval_metric='mlogloss',
                               tree_method='hist', random_state=SEED, n_jobs=1)),
        {'model__n_estimators': [100, 200, 400], 'model__max_depth': [2, 3, 5],
         'model__learning_rate': [0.02, 0.05, 0.1], 'model__min_child_weight': [1, 5, 10],
         'model__subsample': [0.8, 1.0], 'model__colsample_bytree': [0.8, 1.0],
         'model__reg_lambda': [1, 5, 10]})}

searches, comparison = {}, []
for name, (estimator, parameters) in candidates.items():
    search = RandomizedSearchCV(estimator, parameters, n_iter=12, scoring=SCORING,
        refit='f1_macro', cv=cv, random_state=SEED, n_jobs=2, error_score='raise', return_train_score=True)
    search.fit(X_train, y_train)
    searches[name] = search
    result = pd.DataFrame(search.cv_results_)
    result.to_csv(ROOT / f'outputs/tuning_{name.lower().replace(" ", "_")}.csv', index=False)
    i = search.best_index_
    row = {'model': name}
    for metric in SCORING:
        row[metric] = float(result.loc[i, f'mean_test_{metric}'])
        row[f'{metric}_sd'] = float(result.loc[i, f'std_test_{metric}'])
    comparison.append(row)
    print(name, 'CV macro F1:', round(row['f1_macro'], 4))

leaderboard = pd.DataFrame(comparison).sort_values(['f1_macro', 'roc_auc_ovr_macro', 'model'], ascending=[False, False, True])
leaderboard.to_csv(ROOT / 'outputs/cv_comparison.csv', index=False)
selected_name = leaderboard.iloc[0]['model']
selected = searches[selected_name].best_estimator_
assert list(selected.classes_) == [0, 1, 2]
selection = {'model': selected_name, 'criterion': 'development CV macro F1; tie-break macro OVR ROC AUC',
             'parameters': searches[selected_name].best_params_, 'test_used_for_selection': False}
(ROOT / 'outputs/model_selection.json').write_text(json.dumps(selection, indent=2, default=str))
display(leaderboard)
print('Selected:', selected_name)
baseline = pipeline(DummyClassifier(strategy='prior'))
baseline_cv = cross_validate(baseline, X_train, y_train, cv=cv, scoring=SCORING, error_score='raise')
pd.DataFrame({k: v for k, v in baseline_cv.items() if k.startswith('test_')}).to_csv(ROOT / 'outputs/baseline_cv.csv', index=False)
baseline.fit(X_train, y_train)

## 4. Independent test evaluation

In [ ]:
def metrics(truth, probability):
    prediction = probability.argmax(axis=1)
    return {'accuracy': accuracy_score(truth, prediction),
            'precision_macro': precision_score(truth, prediction, average='macro', zero_division=0),
            'recall_macro': recall_score(truth, prediction, average='macro', zero_division=0),
            'f1_macro': f1_score(truth, prediction, average='macro', zero_division=0),
            'roc_auc_ovr_macro': roc_auc_score(truth, probability, multi_class='ovr', average='macro', labels=[0,1,2])}

test_rows = []
for name, estimator in [(n, s.best_estimator_) for n, s in searches.items()] + [('Prior baseline', baseline)]:
    probabilities = estimator.predict_proba(X_test)
    test_rows.append({'model': name, **metrics(y_test, probabilities)})
    report = classification_report(y_test, probabilities.argmax(axis=1), labels=[0,1,2], target_names=LABELS, output_dict=True, zero_division=0)
    pd.DataFrame(report).T.to_csv(ROOT / f'outputs/test_report_{name.lower().replace(" ", "_")}.csv')
test_results = pd.DataFrame(test_rows)
test_results.to_csv(ROOT / 'outputs/test_comparison.csv', index=False)
display(test_results)

probability = selected.predict_proba(X_test)
prediction = probability.argmax(axis=1)
ConfusionMatrixDisplay.from_predictions(y_test, prediction, labels=[0,1,2], display_labels=LABELS, cmap='Blues', xticks_rotation=15)
plt.title(selected_name); plt.tight_layout()
plt.savefig(ROOT / 'report_figures/confusion_matrix.png', dpi=200); plt.show()
fig, ax = plt.subplots(figsize=(6,5))
binary = label_binarize(y_test, classes=[0,1,2])
for k, label in enumerate(LABELS):
    RocCurveDisplay.from_predictions(binary[:, k], probability[:, k], name=label, ax=ax)
ax.plot([0,1],[0,1],'--',color='grey'); ax.set_title('Selected model: one-vs-rest ROC')
plt.tight_layout(); plt.savefig(ROOT / 'report_figures/roc_curves.png', dpi=200); plt.show()

# Resample independent DX-pattern groups; retain all responses within a sampled group.
rng = np.random.default_rng(SEED)
g = groups.iloc[test_idx].to_numpy()
unique_groups = np.unique(g)
bootstrap = []
for _ in range(1000):
    sampled = rng.choice(unique_groups, len(unique_groups), replace=True)
    idx = np.concatenate([np.flatnonzero(g == value) for value in sampled])
    if len(np.unique(y_test.iloc[idx])) == 3:
        bootstrap.append(metrics(y_test.iloc[idx], probability[idx]))
ci = pd.DataFrame(bootstrap).quantile([0.025,0.975]).T
ci.columns = ['lower_95', 'upper_95']
ci.insert(0, 'estimate', pd.Series(metrics(y_test, probability)))
ci.to_csv(ROOT / 'outputs/selected_model_confidence_intervals.csv')
display(ci)

## 5. Explain and export the selected pipeline
SHAP explains class probabilities relative to equal-weight synthetic response levels 0–4. This reference is not an average student. Contributions describe model associations, not causes. No respondent records are included in the deployment artifact.

In [ ]:
import shap
background = pd.DataFrame(np.repeat(np.arange(5)[:,None], 20, axis=1), columns=FEATURES, dtype=float)
def predict_probability(values):
    return selected.predict_proba(pd.DataFrame(values, columns=FEATURES))
explainer = shap.Explainer(predict_probability, background, algorithm='permutation', feature_names=FEATURES, seed=SEED)
explained = explainer(X_test, max_evals=5 * (2 * len(FEATURES) + 1))
reconstructed = explained.base_values + explained.values.sum(axis=1)
np.testing.assert_allclose(reconstructed, probability, atol=1e-6)
global_importance = pd.DataFrame({'feature': FEATURES, 'mean_abs_shap': np.abs(explained.values).mean(axis=(0,2))}).sort_values('mean_abs_shap', ascending=False)
global_importance.to_csv(ROOT / 'outputs/shap_global.csv', index=False)
global_importance.sort_values('mean_abs_shap').plot.barh(x='feature', y='mean_abs_shap', legend=False, figsize=(7,6), color='#287c74')
plt.xlabel('Mean absolute SHAP contribution (probability)'); plt.tight_layout()
plt.savefig(ROOT / 'report_figures/shap_global.png', dpi=200); plt.show()
schema = {'title': 'An Explainable Machine Learning System for Estimating Help-Seeking Intentions from Dating Violence Experiences Among Sri Lankan University Students',
          'features': FEATURES, 'questions': {f: columns[f].split('.',1)[1].strip() for f in FEATURES},
          'labels': LABELS, 'scale': {'0':'Never','1':'Rarely','2':'Sometimes','3':'Often','4':'All of the time'},
          'reference': 'Equal-weight synthetic response levels 0-4; not an average student',
          'target_rule': audit['target_rule'], 'reference_period': 'past six months', 'model': selected_name,
          'test_metrics': metrics(y_test, probability), 'development_n': len(train_idx), 'test_n': len(test_idx),
          'versions': {p: metadata.version(p) for p in ['numpy','pandas','scipy','scikit-learn','xgboost','shap','joblib']},
          'python': platform.python_version(), 'source_sha256': audit['source_sha256'],
          'interpretation': 'Estimates stated professional help-seeking intention; not actual help-seeking, diagnosis, or violence severity.'}
joblib.dump(selected, ROOT / 'models/h3_pipeline.joblib', compress=3)
(ROOT / 'models/schema.json').write_text(json.dumps(schema, indent=2, ensure_ascii=False), encoding='utf-8')
loaded = joblib.load(ROOT / 'models/h3_pipeline.joblib')
np.testing.assert_allclose(loaded.predict_proba(X_test), probability)
(ROOT / 'outputs/environment.json').write_text(json.dumps(schema['versions'], indent=2))
print('Saved selected pipeline, schema, evaluation tables and figures.')